# 02 - Population genetics: PCA, ADMIXTURE, $F_{ST}$, ROH

![cohort](https://img.shields.io/badge/cohort-KZ--219-1b9e77)
![platform](https://img.shields.io/badge/platform-DNBSEQ--G400-7570b3)
![pipeline](https://img.shields.io/badge/DRAGEN-v3.10.16-d95f02)
![reference](https://img.shields.io/badge/reference-GRCh37%2Fhg19-e7298a)
![annotation](https://img.shields.io/badge/ANNOVAR-v2025Mar02-666666)
![bioproject](https://img.shields.io/badge/BioProject-PRJNA1224597-333333)

| | |
|---|---|
| **Cohort** | KZ-219 - 219 healthy Kazakh adults |
| **Sequencing** | DNBSEQ-G400, PCR-free, 100 bp paired-end |
| **Alignment and variant calling** | Illumina DRAGEN Bio-IT Platform v3.10.16 |
| **Reference** | GRCh37 / hg19 (DRAGEN hash table v8) |
| **Annotation** | ANNOVAR v2025Mar02 |
| **Raw data** | NCBI SRA, BioProject PRJNA1224597 |

| Item | File | Content |
|---|---|---|
| **Fig. 2** | `Fig2_PCA_ADMIXTURE` | Eurasian PCA, ADMIXTURE CV and ancestry proportions |
| **Fig. 3** | `Fig3_ROH` | F_ROH by population and by run length |
| **Supplementary Fig. 3** | `Supplementary_Fig_3_tribe_PCA` | within-cohort PCA by tribe |
| **Supplementary Fig. 4** | `Supplementary_Fig_4_AADR_projection` | AADR groups projected onto the reference axes |
| **Supplementary Fig. 5** | `Supplementary_Fig_5_Fst_heatmaps` | pairwise F_ST, both reference panels |
| **Supplementary Tables 2-6** | `Supplementary_Tables_02.docx` | 2 relatedness, 3 structure by tribe and juz, 4 F_ROH by population, 5 reference panels, 6 robustness |
| **Supplementary Data 2** | `Supplementary_Data_2_pca_admixture_fst_roh.xlsx` | all tables, plus source data for every figure panel |

### Running

The notebook reads the outputs of the population-genetics shell pipeline
(`04_pca.sh`, `05_admixture.sh`, `06_fst.sh`, `08_project_aadr.sh` and the ROH and
relatedness steps) from `$KZ_PROJECT/results/`:

```
results/
  groupB_kz_hgdp/        PCA, ADMIXTURE, F_ST, ROH, AADR projection (cohort + HGDP)
  groupA_kz_1000g_sgdp/  F_ST (cohort + 1000 Genomes + SGDP)
  kz_tribes/             within-cohort PCA, PC association, kinship by tribe and juz
  kz_only/               within-cohort ROH
  meta/                  tribe labels, panel QC
```

`figstyle.py` (shared plotting helpers) sits next to the notebook.

```bash
export KZ_PROJECT=data/popgen      
export KZ_OUT=results/02_popgen
pip install -r requirements.txt
jupyter nbconvert --to notebook --execute --inplace 02_population_genetics.ipynb
```


## 0. Setup

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))
import figstyle as fs
from figstyle import (load, nums, figure, panel, finish, save, label_groups,
                      pretty, DISPLAY_NAME,
                      W1, W15, W2, MM, PALETTE, OKABE_ITO, SUPERPOP_COLOUR,
                      COHORT_COLOUR, GREY)

PROJECT = Path(os.environ.get("KZ_PROJECT", "data/popgen"))
RES = PROJECT/"results"
GA  = RES/"groupA_kz_1000g_sgdp"
GB  = RES/"groupB_kz_hgdp"        
KZ  = RES/"kz_tribes"
LEVEL = "nation"                  

OUT = Path(os.environ.get("KZ_OUT", "results/02_popgen"))
OUT.mkdir(parents=True, exist_ok=True)
fs.FIGDIR = OUT

import matplotlib as mpl
from matplotlib.text import Text
from PIL import Image
import pandas as pd

FS, FS_PANEL, FS_MAX = 8, 10, 12          
DPI, FIG_FORMATS = 1200, ("tiff", "png") 
TEXT_DARK = "#4A4A4A"           

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
    "font.size": FS, "axes.labelsize": FS, "axes.titlesize": FS,
    "xtick.labelsize": FS, "ytick.labelsize": FS,
    "legend.fontsize": FS, "legend.title_fontsize": FS,
    "hatch.linewidth": 0.5, "lines.linewidth": 1.0,
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "figure.facecolor": "white", "savefig.facecolor": "white",
})
for _a in ("FS_LABEL", "FS_TICK", "FS_LEGEND", "FS_TEXT", "FS_ANNOT", "FS_SMALL"):
    setattr(fs, _a, FS)
fs.FS_PANEL = FS_PANEL

_fs_panel, _fs_finish = fs.panel, fs.finish


def panel(ax, letter, **kw):
    """figstyle.panel, with the letter forced to 10 pt bold."""
    fig = ax.figure
    before = {id(t) for t in list(ax.texts) + list(fig.texts)}
    out = _fs_panel(ax, letter, **kw)
    for t in list(ax.texts) + list(fig.texts):
        if id(t) not in before:
            t.set_fontsize(FS_PANEL); t.set_fontweight("bold")
    return out


def finish(ax, *a, **k):
    """figstyle.finish, with axis labels and tick labels at 8 pt."""
    out = _fs_finish(ax, *a, **k)
    ax.xaxis.label.set_size(FS); ax.yaxis.label.set_size(FS)
    ax.tick_params(labelsize=FS)
    return out


def _enforce_text(fig):
    """Raise any text below 8 pt (and cap above 12 pt); return what was changed."""
    changed = []
    for t in fig.findobj(Text):
        if not t.get_visible() or not t.get_text().strip():
            continue
        s = t.get_fontsize()
        if s < FS - 1e-6 or s > FS_MAX + 1e-6:
            changed.append((t.get_text()[:30], s))
            t.set_fontsize(min(max(s, FS), FS_MAX))
    return changed


def save(fig, stem, dpi=DPI, formats=FIG_FORMATS):
    """PNG (RGB) and TIFF (CMYK, LZW) at 1200 dpi, named with the figure number."""
    changed = _enforce_text(fig)
    if changed:
        print(f"  NOTE {stem}: {len(changed)} text item(s) raised to {FS} pt, check the layout: "
              + ", ".join(f"'{t}' ({s:g})" for t, s in changed[:6])
              + (" ..." if len(changed) > 6 else ""))
    png = OUT/f"{stem}.png"
    fig.savefig(png, dpi=dpi, bbox_inches="tight", pad_inches=0.03)
    paths = []
    if "tiff" in formats:
        tif = OUT/f"{stem}.tiff"
        with Image.open(png) as im:
            im.convert("RGB").convert("CMYK").save(tif, compression="tiff_lzw", dpi=(dpi, dpi))
        paths.append(tif)
    if "png" in formats:
        paths.append(png)
    else:
        png.unlink()
    for p in paths:
        with Image.open(p) as im:
            w, h, mode = *im.size, im.mode
        print(f"  saved {p.name}: {mode}, {w / dpi * 25.4:.0f} x {h / dpi * 25.4:.0f} mm "
              f"at {dpi} dpi, {p.stat().st_size / 1e6:.1f} MB")
    return paths

SOURCE = {}
_ID_COLS = {"IID", "FID", "iid", "fid", "sample", "Sample", "ID"}


def source(sheet, tbl, cols=None):
    if tbl is None:
        return
    df = tbl.copy() if isinstance(tbl, pd.DataFrame) else pd.DataFrame(dict(tbl))
    df = df[[c for c in df.columns if c not in _ID_COLS]]
    if cols:
        df = df[[c for c in cols if c in df.columns]]
    for c in df.columns:
        conv = pd.to_numeric(df[c], errors="coerce")
        if conv.notna().sum() == df[c].notna().sum():  
            df[c] = conv
    SOURCE[sheet] = df

fs.report_environment()
assert RES.exists(), f"results directory not found: {RES} (set KZ_PROJECT)"
print(f"input : {RES}\noutput: {OUT}")

## Figure 2 - Eurasian principal components, ADMIXTURE

PC1 carries about 5% of the variance and PC2 well under 1%, so a Eurasia-wide
scatter is close to a one-dimensional result and says little on its own. The
content is at population level, which is why **a** labels population centroids
and **b** zooms on the cohort's own neighbourhood. Panel b carries the claim;
panel a is the context it sits in. Panel b has no title inside the figure; the
legend says that it shows the cohort and its nine nearest populations. The CV axis is
labelled "CV error" so that the label fits the panel height at 8 pt.

Superpopulations are separated by marker shape as well as colour.

Trained on the cohort-capped set and projected onto the full panel with
`admixture -P`, so no cohort individual influenced the component definitions.
CV error is the mean over 10 seeds with its standard deviation. The layout is
built from what exists: with no Q table yet, only the CV panel is drawn rather
than leaving an empty half-canvas.

In [ ]:
_S = globals().get("fs")
LW_LN = getattr(_S, "LW_LINE", globals().get("LW_LINE", 1.2))

pc    = load(GB/"pca/pca_eurasia.tsv",           "Fig 2  PCA", required=False)
ve    = load(GB/"pca/pca_eurasia.varexp.tsv",    "Fig 2  variance", required=False)
cv    = load(GB/"admixture/cv_summary.tsv",      "Fig 2  CV", required=False)
modes = load(GB/"admixture/train.modes_by_K.tsv","Fig 2  modes", required=False)

BEST_K = int(cv["K"][int(np.nanargmin(nums(cv["CV_mean"])))]) if cv else None
bk = GB/"admixture/best_k.tsv"
if bk.exists() and BEST_K is not None:
    import re
    _m = re.search(r"best_k\s+(\d+)", bk.read_text())
    if _m and int(_m.group(1)) != BEST_K:
        print(f"  NOTE: best_k.tsv says K={_m.group(1)} but the CV minimum in "
              f"cv_summary.tsv is K={BEST_K}; using {BEST_K}")
Q = load(GB/f"admixture/Q_projected.K{BEST_K}.tsv", f"Fig 2  Q at K={BEST_K}",
         required=False) if BEST_K else None

if pc is None or cv is None:
    print("  SKIP Fig 2: needs 04_pca.sh and 05_admixture.sh")
else:
    x, y = nums(pc["PC1"]), nums(pc["PC2"])
    pops, nats = pc["Population"], pc["Nation"]
    COH = "Kazakh" if "Kazakh" in set(nats) else max(set(nats), key=nats.count)

    def lab(i):
        if ve is None or i not in ve["PC"]:
            return i
        return f"{i} ({float(ve['pct_variance'][ve['PC'].index(i)]):.2f}%)"

    SUPERPOP_MARKER = {"AFR": "^", "OCE": "v", "EUR": "s", "WEA": "D",
                       "SAS": "P", "EAS": "o", "CAS": "X", "KAZ": "o"}

    def draw(ax):
        for p in [q for q in ("AFR", "OCE", "EUR", "WEA", "SAS", "EAS", "CAS",
                              "KAZ") if q in set(pops)]:
            m = np.array([q == p for q in pops]); c_ = (p == "KAZ")
            ax.scatter(x[m], y[m], s=13 if c_ else 7,
                       marker=SUPERPOP_MARKER.get(p, "o"),
                       c=SUPERPOP_COLOUR.get(p, GREY),
                       edgecolors="black" if c_ else "none",
                       linewidths=0.3 if c_ else 0,
                       alpha=0.95 if c_ else 0.55,
                       zorder=3 if c_ else 2, label=p, rasterized=True)

    fig = plt.figure(figsize=(W2, 168*MM))
    gs  = fig.add_gridspec(3, 2, height_ratios=[1.0, 0.52, 0.92],
                           hspace=0.45, wspace=0.20)
    ax_a = fig.add_subplot(gs[0, 0])
    ax_b = fig.add_subplot(gs[0, 1])
    ax_c = fig.add_subplot(gs[1, :])
    ax_d = fig.add_subplot(gs[2, :]) if Q is not None else None

    # a - Eurasian context
    draw(ax_a); finish(ax_a, lab("PC1"), lab("PC2"))
    ax_a.legend(loc="best", ncol=2, handletextpad=0.2, columnspacing=0.6,
                borderpad=0.2, labelspacing=0.25, markerscale=1.6, fontsize=FS)
    shown = pretty(nats)
    big = sorted(set(nats), key=lambda n: -nats.count(n))[:8]
    label_groups(ax_a, x, y, shown, only=set(pretty(list(big) + [COH])),
                 fontsize=FS)

    # b - neighbourhood
    cen = {n: (float(np.nanmean(x[[i for i, v in enumerate(nats) if v == n]])),
               float(np.nanmean(y[[i for i, v in enumerate(nats) if v == n]])))
           for n in set(nats) if nats.count(n) >= 8}
    cx, cy = cen[COH]
    near = sorted(cen, key=lambda n: (cen[n][0]-cx)**2 + (cen[n][1]-cy)**2)[:9]
    sel = np.array([n in near for n in nats])
    padx = 0.18*(np.nanmax(x[sel]) - np.nanmin(x[sel]) + 1e-9)
    pady = 0.18*(np.nanmax(y[sel]) - np.nanmin(y[sel]) + 1e-9)
    draw(ax_b)
    ax_b.set_xlim(np.nanmin(x[sel])-padx, np.nanmax(x[sel])+padx)
    ax_b.set_ylim(np.nanmin(y[sel])-pady, np.nanmax(y[sel])+pady)
    finish(ax_b, lab("PC1"), lab("PC2"))
    label_groups(ax_b, x, y, shown, only=set(pretty(near)), fontsize=FS)

    # c
    K, m, sd = nums(cv["K"]), nums(cv["CV_mean"]), nums(cv["CV_sd"])
    o = np.argsort(K)
    ax_c.errorbar(K[o], m[o], yerr=sd[o], marker="o", ms=3,
                  color=OKABE_ITO["blue"], lw=LW_LN, elinewidth=1.0,
                  capsize=1.5)
    if BEST_K:
        ax_c.axvline(BEST_K, color=COHORT_COLOUR, lw=1.0, ls="--", zorder=1)
    ax_c.set_xticks(K[o].astype(int))
    finish(ax_c, "K", "CV error")
    if modes is not None:
        for kk, fr in zip(nums(modes["K"]), nums(modes["major_mode_fraction"])):
            j = np.where(K[o] == kk)[0]
            if j.size:
                ax_c.annotate(f"{fr:.0%}", (kk, m[o][j[0]]), fontsize=FS,
                              color=TEXT_DARK, ha="center",
                              textcoords="offset points", xytext=(0, 7))

    # d
    if Q is not None:
        qc = sorted([c for c in Q if c.startswith("Q") and c[1:].isdigit()],
                    key=lambda c: int(c[1:]))
        Qm  = np.column_stack([nums(Q[c]) for c in qc])
        grp = Q["Nation"] if "Nation" in Q else Q["Population"]
        rk = load(GB/f"fst/{LEVEL}/fst_vs_cohort.tsv", "Fig 2  ordering",
                  required=False)
        QCOH = "Kazakh" if "Kazakh" in set(grp) else max(set(grp), key=grp.count)
        if rk is not None:
            near_r = {p: i for i, p in enumerate(rk["population"])}
            meanQ = {g: Qm[[i for i, v in enumerate(grp) if v == g]].mean(0)
                     for g in set(grp)}
            qd = {g: float(np.linalg.norm(meanQ[g] - meanQ[QCOH])) for g in meanQ}
            ranked = sorted((g for g in meanQ if g in near_r),
                            key=lambda g: near_r[g])
            def key(g):
                if g == QCOH:   return (0, 0.0)
                if g in near_r: return (1, float(near_r[g]))
                below = sum(1 for r in ranked if qd[r] < qd[g])
                return (1, below - 0.5)
            gg = sorted(set(grp), key=lambda g: key(g) + (g,))
            unranked = [g for g in gg if g not in near_r and g != QCOH]
            print(f"  ordered by Fst to {QCOH}: " + ", ".join(gg[1:6]) + ", ...")
            if unranked:
                print(f"  placed by ancestry-profile distance (absent from the "
                      f"Fst table): {', '.join(unranked)}")
        else:
            gg = sorted(set(grp), key=lambda g: (
                g != QCOH,
                int(np.argmax(Qm[[i for i, v in enumerate(grp) if v == g]].mean(0)))))
        idx = []
        for g in gg:
            ii = [i for i, v in enumerate(grp) if v == g]
            d  = int(np.argmax(Qm[ii].mean(0)))
            idx += sorted(ii, key=lambda i: -Qm[i, d])
        Qs, seen = Qm[idx], [grp[i] for i in idx]
        bottom = np.zeros(len(idx))
        for j in range(Qs.shape[1]):
            ax_d.bar(np.arange(len(idx)), Qs[:, j], bottom=bottom, width=1.0,
                     color=PALETTE[j % len(PALETTE)], linewidth=0)
            bottom += Qs[:, j]
        ax_d.set_xlim(-0.5, len(idx)-0.5); ax_d.set_ylim(0, 1)
        pos = []
        for g in gg:
            w = [i for i, v in enumerate(seen) if v == g]
            pos.append((g, float(np.mean(w))))
            if w[0] > 0:
                ax_d.axvline(w[0]-0.5, color="white", lw=1.0)
        ax_d.set_yticks([0, 0.5, 1])
        finish(ax_d, None, f"ancestry proportion (K = {BEST_K})")
        ax_d.spines["left"].set_visible(True)

        from matplotlib.transforms import blended_transform_factory
        LEFT_F, RIGHT_F, FIG_H_MM = 0.105, 0.99, 168.0
        LEAD_MM, GAP_MM = 3.0, 3.0        
        ax_w_mm = W2 / MM * (RIGHT_F - LEFT_F)
        unit_mm = ax_w_mm / len(idx)             
        cen = np.array([p for _, p in pos])
        n_lab = len(cen)
        gap_mm = min(GAP_MM, ax_w_mm / n_lab)
        s = gap_mm / unit_mm         

        def _pava(v):
            """Non-decreasing least-squares fit (pool adjacent violators)."""
            blocks = []
            for x_ in v:
                blocks.append([x_, 1])
                while len(blocks) > 1 and blocks[-2][0] > blocks[-1][0]:
                    m2, n2 = blocks.pop(); m1, n1 = blocks.pop()
                    blocks.append([(m1 * n1 + m2 * n2) / (n1 + n2), n1 + n2])
            return np.concatenate([[m_] * n_ for m_, n_ in blocks])

        k_ = np.arange(n_lab)
        lo_, hi_ = -0.5 + s / 2, len(idx) - 0.5 - s / 2
        q = np.clip(_pava(cen - k_ * s), lo_, hi_ - (n_lab - 1) * s)
        lab_x = q + k_ * s

        names = pretty([g for g, _ in pos])
        _tmp = [ax_d.text(0, 0, t_, rotation=90, fontsize=FS,
                          fontweight="bold" if g == QCOH else "normal")
                for t_, (g, _) in zip(names, pos)]
        fig.canvas.draw()
        _r = fig.canvas.get_renderer()
        lab_mm = max(t_.get_window_extent(_r).height for t_ in _tmp) / fig.dpi * 25.4
        for t_ in _tmp:
            t_.remove()
        bot = min(0.30, (lab_mm + LEAD_MM + 6) / FIG_H_MM)
        fig.subplots_adjust(left=LEFT_F, right=RIGHT_F, top=0.955, bottom=bot)
        ax_h_mm = ax_d.get_position().height * FIG_H_MM

        ax_d.set_xticks([])
        blend = blended_transform_factory(ax_d.transData, ax_d.transAxes)
        _labels = []
        for (g, c0), x_, t_ in zip(pos, lab_x, names):
            is_coh = g == QCOH
            a_ = ax_d.annotate(
                t_, xy=(c0, 0), xycoords=blend,
                xytext=(x_, -LEAD_MM / ax_h_mm), textcoords=blend,
                rotation=90, ha="center", va="top", fontsize=FS,
                color=COHORT_COLOUR if is_coh else "black",
                fontweight="bold" if is_coh else "normal",
                annotation_clip=False,
                arrowprops=dict(arrowstyle="-", lw=0.4, color=TEXT_DARK,
                                shrinkA=0, shrinkB=0, relpos=(0.5, 1.0)))
            _labels.append(a_)

        fig.canvas.draw()
        _r = fig.canvas.get_renderer()
        from matplotlib.text import Text as _Text
        _bb = [_Text.get_window_extent(a_, _r) for a_ in _labels]   # text only, not the leader
        _tol = 0.3 / 25.4 * fig.dpi                # 0.3 mm: touching em boxes are not an overlap
        _ov = sum(b.x0 < a.x1 - _tol for a, b in zip(_bb, _bb[1:]))
        _shift = np.abs(lab_x - cen) * unit_mm
        print(f"  Fig 2d: {n_lab} group labels at {FS} pt, spacing {gap_mm:.2f} mm, "
              f"largest shift {_shift.max():.1f} mm, "
              + (f"{_ov} adjacent pairs OVERLAP" if _ov else "no overlaps"))
        if gap_mm < 2.4:
            print("  NOTE: too many groups for 8 pt labels across 180 mm")
        source("SD_Fig2d_ADMIXTURE_Q", Q)
    else:
        print("  Q table absent: panel d omitted")
        bot = 0.06

    for ax, L in zip([ax_a, ax_b], "ab"):
        panel(ax, L, dx=-0.13)
    panel(ax_c, "c", dx=-0.115)
    if ax_d is not None:
        panel(ax_d, "d", dx=-0.115)
    fig.subplots_adjust(left=0.105, right=0.99, top=0.955, bottom=bot)
    source("SD_Fig2ab_PCA", pc)
    _cvsd = pd.DataFrame(dict(cv))
    if modes is not None:
        _cvsd = _cvsd.merge(pd.DataFrame(dict(modes)), on="K", how="left",
                            suffixes=("", "_modes"))
    source("SD_Fig2c_CV", _cvsd)
    save(fig, "Fig2_PCA_ADMIXTURE"); plt.show()

## Supplementary Fig. 5 - Pairwise $F_{ST}$

Hudson's estimator with a block-jackknife standard error. Matrix rows and
columns are ordered by average-linkage clustering with optimal leaf ordering.
**a** is the HGDP merge, **b** the 1000 Genomes + SGDP merge, on one shared
colour scale. The two matrices are stacked rather than placed side by side so
that the population labels can be set at 8 pt; the cell size is the largest that
keeps the figure on one A4 page, and the panel names that used to be titles
inside the figure go to the legend. The cohort-against-everyone ranking and
`SEPARATED_FROM_PREVIOUS` are in Supplementary Data 2, sheet `2_Fst_vs_cohort`.

In [ ]:
# Supplementary Fig. 5 - pairwise Fst on both panels, one shared scale
#FST_MIN_N_NATION=5
from matplotlib.colors import BoundaryNorm

_S = globals().get("fs")
FS_LAB = FS_TIC = FS_LEG = FS
LW_AX  = getattr(_S, "LW_AXIS",   globals().get("LW_AXIS", 1.0))

DISPLAY_NAME.update({
    "CHB": "Han (Beijing)",     "CHS": "Han (South)",
    "JPT": "Japanese",          "CDX": "Dai",           "KHV": "Kinh",
    "CEU": "European (Utah)",   "TSI": "Tuscan",        "FIN": "Finnish",
    "GBR": "British",           "IBS": "Iberian",
    "GIH": "Gujarati",          "PJL": "Punjabi",       "BEB": "Bengali",
    "STU": "Tamil (Sri Lanka)", "ITU": "Telugu",
    "YRI": "Yoruba",            "LWK": "Luhya",         "GWD": "Gambian",
    "MSL": "Mende",             "ESN": "Esan",
    "ASW": "African American",  "ACB": "African Caribbean",
    "MXL": "Mexican",           "PUR": "Puerto Rican",
    "CLM": "Colombian",         "PEL": "Peruvian",
})

FST_LEVELS = [0, 0.002, 0.005, 0.01, 0.02, 0.05, 0.10, 0.20, 0.35]

PANEL_NAMES = {"kz": "cohort", "1000g": "1000G", "sgdp": "SGDP",
               "hgdp": "HGDP 650k", "aadr": "AADR"}

def panels_of(group_dir):
    parts = Path(group_dir).name.split("_")[1:]
    return " + ".join(PANEL_NAMES.get(p, p) for p in parts)

def _panel(ax, mat, norm, cmap, group, which):
    labs = mat["POP"]
    Mx = np.array([[float(mat[c][i]) for c in labs] for i in range(len(labs))])
    np.fill_diagonal(Mx, np.nan)
    n = len(labs)
    im = ax.imshow(Mx, cmap=cmap, norm=norm, interpolation="nearest")
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(with_n(labs, which), rotation=90, fontsize=FS)
    ax.set_yticklabels(with_n(labs, which), fontsize=FS)
    for coh in ("Kazakh", "KAZ"):
        if coh in labs:
            k = labs.index(coh)
            for t in (ax.get_xticklabels()[k], ax.get_yticklabels()[k]):
                t.set_color(COHORT_COLOUR); t.set_fontweight("bold")
            break
    ax.tick_params(length=1.2, pad=1)
    for sp in ax.spines.values():
        sp.set_visible(False)
    return im

HEAT_LEVEL = LEVEL
for suffix in ("_minn5", "_minn8"):
    if (GB/f"fst/{LEVEL}{suffix}/fst_matrix_ordered.tsv").exists() and \
       (GA/f"fst/{LEVEL}{suffix}/fst_matrix_ordered.tsv").exists():
        HEAT_LEVEL = LEVEL + suffix
        break

matB = load(GB/f"fst/{HEAT_LEVEL}/fst_matrix_ordered.tsv", "Suppl Fig 5a  groupB", required=False)
matA = load(GA/f"fst/{HEAT_LEVEL}/fst_matrix_ordered.tsv", "Suppl Fig 5b  groupA", required=False)
rank = load(GB/f"fst/{LEVEL}/fst_vs_cohort.tsv",      "Suppl Fig 5  ranking", required=False)

# n per population, PER GROUP
def _n_of(g):
    t_ = load(g/f"fst/{HEAT_LEVEL}/pop_n.tsv", "", required=False)
    return {p_: int(v) for p_, v in zip(t_["POP"], t_["N"])} if t_ else {}
_N_BY_GROUP = {"B": _n_of(GB), "A": _n_of(GA)}

MAIN_MIN_N = None
_kept = load(GB/f"fst/{LEVEL}/fst_vs_cohort.tsv", "", required=False)
if _kept and _kept.get("N"):
    MAIN_MIN_N = min(int(v) for v in _kept["N"])

def with_n(labels, which):
    """Print n on any population that would not have passed the main threshold."""
    out = []
    nmap = _N_BY_GROUP.get(which, {})
    for l in labels:
        n_ = nmap.get(l)
        out.append(f"{pretty(l)} ({n_})"
                   if (MAIN_MIN_N and n_ and n_ < MAIN_MIN_N) else pretty(l))
    return out

if matB is None or matA is None:
    print("  need fst_matrix_ordered.tsv for BOTH groups at this level "
          "- run 06_fst.sh")
else:
    cmap = plt.get_cmap("viridis", len(FST_LEVELS) - 1)
    norm = BoundaryNorm(FST_LEVELS, cmap.N)
    nB, nA = len(matB["POP"]), len(matA["POP"])

    W_MM, H_MAX_MM, CELL_MAX_MM = W2 / MM, 245.0, 3.4
    TOP, GAP, BOT, PAD = 7.0, 9.0, 2.0, 1.5
    fig = plt.figure(figsize=(W2, H_MAX_MM * MM))
    axB = fig.add_axes([0.3, 0.55, 0.4, 0.4])
    axA = fig.add_axes([0.3, 0.05, 0.3, 0.3])
    im = _panel(axB, matB, norm, cmap, panels_of(GB), "B")
    _panel(axA, matA, norm, cmap, panels_of(GA), "A")
    fig.canvas.draw()
    rend = fig.canvas.get_renderer()
    px_mm = 25.4 / fig.dpi
    ylab = max(t.get_window_extent(rend).width for ax in (axB, axA)
               for t in ax.get_yticklabels()) * px_mm
    xlabB = max(t.get_window_extent(rend).height for t in axB.get_xticklabels()) * px_mm
    xlabA = max(t.get_window_extent(rend).height for t in axA.get_xticklabels()) * px_mm
    fixed = TOP + xlabB + PAD + GAP + xlabA + PAD + BOT
    cell_mm = min(CELL_MAX_MM, (W_MM - ylab - PAD - 2.0) / nB,
                  (H_MAX_MM - fixed) / (nB + nA))
    H_mm = fixed + (nB + nA) * cell_mm
    fig.set_size_inches(W2, H_mm * MM)
    x0 = ylab + PAD

    def _put(ax, top_mm, n):
        ax.set_position([x0 / W_MM, 1 - (top_mm + n * cell_mm) / H_mm,
                         n * cell_mm / W_MM, n * cell_mm / H_mm])

    _put(axB, TOP, nB)
    topA = TOP + nB * cell_mm + xlabB + PAD + GAP
    _put(axA, topA, nA)
    axes = [axB, axA]

    cb_h = min(55.0, nA * cell_mm)
    cb_x = x0 + nA * cell_mm + 14.0
    cax = fig.add_axes([cb_x / W_MM, 1 - (topA + cb_h) / H_mm, 3.2 / W_MM, cb_h / H_mm])
    cb = fig.colorbar(im, cax=cax, orientation="vertical",
                      spacing="uniform", ticks=FST_LEVELS)
    cb.set_label("Hudson $F_{ST}$", fontsize=FS, labelpad=3)
    cb.ax.set_yticklabels([f"{v:g}" for v in FST_LEVELS], fontsize=FS)
    cb.ax.tick_params(length=1.5, pad=1.5)
    cb.outline.set_linewidth(LW_AX)

    fig.canvas.draw()
    for ax, L in zip(axes, "ab"):
        ax.annotate(L, xy=(0, 1), xycoords="axes fraction",
                    xytext=(-(ylab + PAD) / 25.4 * 72, 2),
                    textcoords="offset points",
                    fontsize=FS_PANEL, fontweight="bold", va="bottom", ha="left")

    source("SD_SFig5a_Fst_HGDP", matB)
    source("SD_SFig5b_Fst_1000G_SGDP", matA)
    save(fig, "Supplementary_Fig_5_Fst_heatmaps")
    plt.show()
    print(f"  cell {cell_mm:.2f} mm, y-labels {ylab:.0f} mm, x-labels {max(xlabA, xlabB):.0f} mm, "
          f"canvas 180 x {H_mm:.0f} mm; a = {panels_of(GB)} ({nB} pops), "
          f"b = {panels_of(GA)} ({nA} pops)")
    if cell_mm < FS * 25.4 / 72:
        print(f"  NOTE: cell {cell_mm:.2f} mm is below the {FS} pt line height "
              f"({FS * 25.4 / 72:.2f} mm); neighbouring labels may touch")

    par = load(RES/"provenance/parameters.tsv", "", required=False)
    key = "FST_MIN_N_NATION" if LEVEL == "nation" else "FST_MIN_N_POP"
    thr = "?"
    if par and key in par.get("parameter", []):
        thr = par["value"][par["parameter"].index(key)]
    if HEAT_LEVEL != LEVEL:
        small = ([l for l in matB["POP"] if MAIN_MIN_N
                  and _N_BY_GROUP["B"].get(l, 10**6) < MAIN_MIN_N] +
                 [l for l in matA["POP"] if MAIN_MIN_N
                  and _N_BY_GROUP["A"].get(l, 10**6) < MAIN_MIN_N])
        print(f"  heatmap drawn from {HEAT_LEVEL}, the tables from "
              f"{LEVEL}: {len(small)} populations here are below n={MAIN_MIN_N} "
              f"and carry their n on the label ({', '.join(sorted(small))})")
    for g, tag in ((GB, "panel a"), (GA, "panel b")):
        ex = load(g/f"fst/{HEAT_LEVEL}/pop_excluded.tsv", "", required=False)
        if ex and ex["POP"]:
            mx = max(int(v) for v in ex["N"])
            print(f"  {tag} ({panels_of(g)}): {len(ex['POP'])} populations "
                  f"merged but excluded at n<{thr}; the largest had n={mx}")

## Figure 3 - Runs of homozygosity

Run-length classes in **b** are separated by hatching as well as colour; tick labels
carry n per population group.

In [ ]:
indiv = load(GB/"roh/froh_individual.tsv",          "Fig 3a,b  FROH", required=False)
kzf   = load(RES/"kz_only/roh/froh_individual.tsv", "Fig 3c  cohort FROH", required=False)

FROH_TOT = FROH_LONG = None
if kzf is not None and "FROH_LONG" in kzf:
    FROH_TOT, FROH_LONG = nums(kzf["FROH"]), nums(kzf["FROH_LONG"])

if indiv is not None:
    fig, axes = plt.subplots(1, 2, figsize=(W2 * 2/3, 70*MM))   
    pops, froh = indiv["Population"], nums(indiv["FROH"])
    order = sorted(set(pops), key=lambda p: -np.nanmedian(
        froh[[i for i,x in enumerate(pops) if x == p]]))
    data = [froh[[i for i,x in enumerate(pops) if x == p]] for p in order]
    bp = axes[0].boxplot(data, positions=range(len(order)), widths=0.62,
                         showfliers=False, patch_artist=True)
    for e in ("boxes","whiskers","caps","medians"):
        for a in bp[e]: a.set_linewidth(1.0); a.set_color("black")
    for b, p in zip(bp["boxes"], order):
        b.set_facecolor(SUPERPOP_COLOUR.get(p, GREY)); b.set_alpha(0.85)
    rng = np.random.default_rng(0)
    for i, d in enumerate(data):
        axes[0].scatter(i + rng.uniform(-0.17, 0.17, len(d)), d, s=1.4,
                        color="black", alpha=0.4, zorder=3, rasterized=True)
    axes[0].set_xticks(range(len(order)))
    nlab = [f"{p} ({len(d)})" for p, d in zip(order, data)]
    axes[0].set_xticklabels(nlab, rotation=90, fontsize=FS)
    axes[0].set_ylim(0, None)
    finish(axes[0], None, r"$F_{ROH}$ (runs $\geq$ 500 kb)")

    cls = [c for c in indiv
           if c.startswith("FROH_") and c != "FROH_LONG"
           and (c.endswith("kb") or c.endswith("kb_plus"))]
    if cls:
        HATCH = ["", "////", "....", "xxxx", "\\\\\\\\", "++++"]
        means_tbl = {"Population": order, "n": [len(d) for d in data]}
        bottom = np.zeros(len(order))
        for j, c in enumerate(cls):
            vals = nums(indiv[c])
            mean = np.array([np.nanmean(vals[[i for i,x in enumerate(pops)
                             if x == p]]) for p in order])
            axes[1].bar(range(len(order)), mean, bottom=bottom, width=0.7,
                        color=PALETTE[j % len(PALETTE)], linewidth=0,
                        hatch=HATCH[j % len(HATCH)], edgecolor="white",
                        label=c[5:].replace("_", "-").replace("kb-plus", "kb+"))
            means_tbl[f"mean {c}"] = mean
            bottom += mean
        assert bottom.max() <= 1.0, "stacked FROH exceeds 1: wrong columns picked"
        axes[1].set_xticks(range(len(order)))
        axes[1].set_xticklabels(nlab, rotation=90, fontsize=FS)
        axes[1].set_ylim(0, bottom.max() * 1.55)    # head-room for the legend
        axes[1].legend(fontsize=FS, loc="upper right", title="run length",
                       title_fontsize=FS, handlelength=1.4, labelspacing=0.3)
        source("SD_Fig3b_FROH_by_class", pd.DataFrame(means_tbl))
        finish(axes[1], None, r"mean $F_{ROH}$ by run length")

    for ax, L in zip(axes, "ab"): panel(ax, L, dx=-0.27)
    fig.tight_layout(w_pad=1.8)
    source("SD_Fig3a_FROH_individual", indiv)
    save(fig, "Fig3_ROH"); plt.show()

## Supplementary Fig. 3 - tribal structure in the cohort PCA

The design is the one that works for a null: **a** puts the whole cohort in
grey behind every panel and highlights one group at a time with its 95%
confidence ellipse and centroid, so "each group sits on top of all the others"
is something the reader sees rather than something the text asserts. A single
scatter with five overlaid colours cannot show that.

**b** collapses it: every group centroid on one axis with a bootstrap interval.
Overlapping intervals across every tribe and zhuz is the result, stated in the
form a reviewer can check.

Tribe labels come from `meta/kz_219_tribes.tsv`, joined to the PC table on IID.

In [ ]:
from matplotlib.patches import Ellipse
from matplotlib.gridspec import GridSpecFromSubplotSpec

tpc  = load(KZ/"pca/pca_unrelated.tsv",     "Suppl Fig 3  cohort PCA", required=False)
tve  = load(KZ/"pca/pca_unrelated.varexp.tsv","Suppl Fig 3  variance", required=False)
trb  = load(RES/"meta/kz_219_tribes.tsv",   "Suppl Fig 3  tribe labels", required=False)
MIN_TRIBE, MIN_ZHUZ, NCOL = 8, 10, 3
PANEL_KEYS = ("Tribe",)
CENT_W_MM, CENT_H_MM, GRID_CELL_MM = 96, 54, 50
TOP_MM, MID_MM, BOT_MM = 6, 13, 13

def col_named(tbl, want):
    for c in tbl:
        if c.strip().lower().replace("_", "") == want.lower():
            return c
    return None

if tpc is not None and trb is not None:
    idc = col_named(trb, "IID") or list(trb)[0]
    lab = {}
    for key in ("Tribe", "Zhuz"):
        c = col_named(trb, key)
        lab[key] = dict(zip(trb[idc], trb[c])) if c else {}

    iid = tpc["IID"]
    x, y = nums(tpc["PC1"]), nums(tpc["PC2"])
    def pct(k):
        if tve is None or f"PC{k}" not in tve["PC"]: return f"PC{k}"
        return f"PC{k} ({float(tve['pct_variance'][tve['PC'].index(f'PC{k}')]):.2f}%)"

    groups = []
    for key, minn in (("Tribe", MIN_TRIBE), ("Zhuz", MIN_ZHUZ)):
        vals = [lab[key].get(i, "NA") for i in iid]
        for lv in sorted({v for v in vals if v not in ("", "NA", "nan")},
                         key=lambda v: -vals.count(v)):
            if vals.count(lv) >= minn:
                groups.append((key, lv, np.array([v == lv for v in vals])))

    COL = {(k, l): PALETTE[i % len(PALETTE)] for i, (k, l, _) in enumerate(groups)}
    panels = [g for g in groups if g[0] in PANEL_KEYS]
    print(f"  groups: {len(groups)} "
          f"({sum(1 for g in groups if g[0]=='Tribe')} tribes, "
          f"{sum(1 for g in groups if g[0]=='Zhuz')} zhuz); drawn: {len(panels)}")

    def ellipse(ax, xs, ys, color, nstd=2.4477):
        if len(xs) < 5: return
        w, v = np.linalg.eigh(np.cov(xs, ys)); o = w.argsort()[::-1]
        w, v = w[o], v[:, o]
        ang = np.degrees(np.arctan2(*v[:, 0][::-1]))
        for fc, al, z in ((color, 0.12, 2), ("none", 1.0, 4)):
            ax.add_patch(Ellipse((xs.mean(), ys.mean()),
                                 *(2*nstd*np.sqrt(w)), angle=ang, facecolor=fc,
                                 edgecolor=color, lw=1.0, ls="--", alpha=al, zorder=z))

    def centroid_ci(m, nboot=2000, seed=0):
        rng = np.random.default_rng(seed)
        X = np.column_stack([x[m], y[m]])
        b = np.array([X[rng.integers(0, len(X), len(X))].mean(0)
                      for _ in range(nboot)])
        return X.mean(0), np.percentile(b, [2.5, 97.5], axis=0)

    nrow = max(1, int(np.ceil(len(panels) / NCOL)))
    grid_h = min(GRID_CELL_MM * nrow,
                 170 - TOP_MM - CENT_H_MM - MID_MM - BOT_MM)
    fig_h_mm = TOP_MM + CENT_H_MM + MID_MM + grid_h + BOT_MM
    fig = plt.figure(figsize=(W2, fig_h_mm * MM))
    outer = fig.add_gridspec(2, 1, height_ratios=[CENT_H_MM, grid_h],
                             hspace=MID_MM / ((CENT_H_MM + grid_h) / 2),
                             left=0.065, right=0.995,
                             top=1 - TOP_MM/fig_h_mm, bottom=BOT_MM/fig_h_mm)
    row_a = GridSpecFromSubplotSpec(1, 2, subplot_spec=outer[0, 0],
                                    width_ratios=[CENT_W_MM, W2/MM - CENT_W_MM],
                                    wspace=0.04)

    axA = fig.add_subplot(row_a[0, 0])
    for key, lv, m in panels:
        c = COL[(key, lv)]
        mu, ci = centroid_ci(m)
        axA.errorbar(mu[0], mu[1],
                     xerr=[[mu[0]-ci[0,0]], [ci[1,0]-mu[0]]],
                     yerr=[[mu[1]-ci[0,1]], [ci[1,1]-mu[1]]],
                     fmt="s", ms=4, mfc=c, mec="black",
                     mew=0.5, ecolor=c, elinewidth=1.1, capsize=2, zorder=3)
        axA.annotate(lv, (mu[0], ci[1, 1]), textcoords="offset points",
                     xytext=(0, 4), ha="center", va="bottom",
                     fontsize=FS, color=TEXT_DARK, fontweight="bold")
    axA.margins(0.22)
    axA.axhline(0, lw=1.0, color="#DDDDDD", zorder=1)
    axA.axvline(0, lw=1.0, color="#DDDDDD", zorder=1)
    finish(axA, "PC1 centroid", "PC2 centroid")

    inner = GridSpecFromSubplotSpec(nrow, NCOL, subplot_spec=outer[1, 0],
                                    wspace=0.16, hspace=0.34)
    qx = np.nanpercentile(x, [1, 99]); qy = np.nanpercentile(y, [1, 99])
    px, py = 0.22*(qx[1]-qx[0]), 0.22*(qy[1]-qy[0])
    xl = (qx[0]-px, qx[1]+px); yl = (qy[0]-py, qy[1]+py)

    grid_axes, ax0 = [], None
    for i, (key, lv, m) in enumerate(panels):
        r, cc = divmod(i, NCOL)
        ax = fig.add_subplot(inner[r, cc], sharex=ax0, sharey=ax0)
        if ax0 is None:
            ax0 = ax
        c = COL[(key, lv)]
        ax.scatter(x, y, s=4, c=GREY, alpha=0.45, linewidths=0, rasterized=True)
        ax.scatter(x[m], y[m], s=13, c=c, alpha=0.95, edgecolors="white",
                   linewidths=0.25, zorder=3, rasterized=True)
        ellipse(ax, x[m], y[m], c)
        ax.scatter([x[m].mean()], [y[m].mean()], s=42, c=c, marker="X",
                   edgecolors="black", linewidths=0.6, zorder=6)
        ax.set_title(f"{lv} (n={int(m.sum())})", pad=2.0,
                     fontsize=FS, color=TEXT_DARK, loc="left")
        ax.set_xlim(xl); ax.set_ylim(yl)
        if i + NCOL < len(panels):
            ax.tick_params(labelbottom=False)
        else:
            ax.set_xlabel(pct(1), fontsize=FS)
        if cc:
            ax.tick_params(labelleft=False)
        else:
            ax.set_ylabel(pct(2), fontsize=FS)
        grid_axes.append(ax)
    for j in range(len(panels), nrow * NCOL):
        r, cc = divmod(j, NCOL)
        fig.add_subplot(inner[r, cc]).axis("off")

    fig.add_subplot(row_a[0, 1]).axis("off")
    axA.annotate("a", (-0.12, 1.04), xycoords="axes fraction", fontsize=FS_PANEL,
                 fontweight="bold", va="bottom", ha="left")
    grid_axes[0].annotate("b", (-0.12, 1.06), xycoords="axes fraction",
                          fontsize=FS_PANEL, fontweight="bold",
                          va="bottom", ha="left")
    _sd = {"PC1": x, "PC2": y,
           "Tribe": [lab["Tribe"].get(i, "NA") for i in iid],
           "Zhuz": [lab["Zhuz"].get(i, "NA") for i in iid]}
    source("SD_SFig3_tribe_PCA", pd.DataFrame(_sd))
    save(fig, "Supplementary_Fig_3_tribe_PCA"); plt.show()

## Supplementary Fig. 4 - AADR projection

Each supplementary figure has to earn its place by showing something a number
cannot: **Supplementary Fig. 4** shows where the AADR groups, unavailable as
diploid genomes, fall on the reference axes.

The cohort-capping check is deliberately NOT a figure. Its nine values are all
pinned at 1.0, so the bars carried no information the numbers did not; they are
in Supplementary Data 2, sheet `8_robustness`, and Supplementary Table 6 (b, c), and one
sentence in the text ("axis correlation |r| > 0.995 for PC1-PC2, subspace overlap 0.999 over
the first six axes") does the whole job.

Drawn at 180 mm instead of 120 mm so that the group labels fit at 8 pt.

In [ ]:
import re
import matplotlib.patheffects as pe
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

AADR_DROP = re.compile(r"_o(\.[A-Z]+)?$")

aadr  = load(GB/"pca_proj_aadr/scores.tsv",            "Suppl Fig 4  AADR", required=False)
amiss = load(GB/"pca_proj_aadr/projected_samples.tsv", "Suppl Fig 4  F_MISS", required=False)

if aadr is None or amiss is None:
    print("  SKIP Suppl Fig 4: run 08_project_aadr.sh")
else:
    keep = [i for i, (n, sr) in enumerate(zip(aadr["Nation"], aadr["Source"]))
            if not (sr == "AADR" and AADR_DROP.search(n))]
    aadr = {k: [v[i] for i in keep] for k, v in aadr.items()}

    fm = {i: float(f) for i, f in zip(amiss["IID"], amiss["F_MISS"])}
    src, nat, pop = aadr["Source"], aadr["Nation"], aadr["Population"]
    x, y = nums(aadr["PC1"]), nums(aadr["PC2"])
    ph   = np.array([s == "AADR" for s in src])
    coh  = np.array([p == "KAZ" for p in pop]) & ~ph
    ref  = ~ph & ~coh
    miss = np.array([fm.get(i, np.nan) for i in aadr["IID"]])
    dip_names = {n for n, p_ in zip(nat, ph) if not p_}
    short = [(n.replace(".SG", "") +
              (" (AADR)" if n.replace(".SG", "") in dip_names else ""))
             if p_ else n
             for n, p_ in zip(nat, ph)]
    aadr_groups = {g for g, p_ in zip(short, ph) if p_}

    groups = sorted(aadr_groups,
                    key=lambda g: np.mean(x[[i for i, s in enumerate(short)
                                             if s == g and ph[i]]]))
    H = (34 + 5.2 * len(groups)) * MM
    fig, axes = plt.subplots(2, 1, figsize=(W2, min(170*MM, 62*MM + H)),
                             gridspec_kw={"height_ratios": [62*MM, H],
                                          "hspace": 0.30})

    # a
    ax = axes[0]
    ax.scatter(x[ref], y[ref], s=4, color=GREY, alpha=0.30, edgecolors="none",
               label="diploid reference", rasterized=True, zorder=1)
    ax.scatter(x[coh], y[coh], s=7, color=COHORT_COLOUR, alpha=0.8,
               edgecolors="none", rasterized=True, zorder=2,
               label="Kazakh cohort")
    for sel, fc, lb in ((ph & (miss <= 0.5), OKABE_ITO["orange"],
                         r"AADR, $F_{MISS}\leq$0.5"),
                        (ph & (miss > 0.5), "none",
                         r"AADR, $F_{MISS}>$0.5")):
        if sel.any():
            ax.scatter(x[sel], y[sel], s=16, facecolors=fc, edgecolors="black",
                       linewidths=0.6, label=lb, zorder=4)
    focus = ph | coh
    fx0, fx1 = float(np.nanmin(x[focus])), float(np.nanmax(x[focus]))
    fy0, fy1 = float(np.nanmin(y[focus])), float(np.nanmax(y[focus]))
    mx, span = 0.10*(fx1-fx0 + 1e-9), (fy1-fy0 + 1e-9)
    ax.set_xlim(fx0-mx, fx1+mx)
    ax.set_ylim(fy0 - 0.50*span, fy1 + 0.95*span)

    ymid = float(np.median(y[ph]))
    cx_lo, cx_hi = np.percentile(x[coh], [2, 98])
    coh_top = float(np.percentile(y[coh], 98))
    over_cohort = (x >= cx_lo) & (x <= cx_hi)
    lift = np.where(over_cohort, (coh_top - y) + 0.26*span, 0.30*span)
    y_lab = np.where((y >= ymid) | over_cohort, y + lift, y - 0.18*span)
    label_groups(ax, x, y_lab, short, only=aadr_groups, fontsize=FS,
                 color={g: COHORT_COLOUR for g in aadr_groups
                        if g.startswith("Kazakh")} or "black")

    for _ln in list(ax.lines):
        if _ln.get_zorder() == 9 and _ln.get_color() == "0.55":
            _ln.remove()

    fig.canvas.draw()
    _r = fig.canvas.get_renderer()
    _lab = [t for t in ax.texts if t.get_text() in aadr_groups]
    _hit = [(a.get_text(), b.get_text())
            for i, a in enumerate(_lab) for b in _lab[i+1:]
            if a.get_window_extent(_r).overlaps(b.get_window_extent(_r))]
    print(f"  Suppl Fig 4a: {len(_lab)} labels, "
          + (f"OVERLAPPING: {_hit}" if _hit else "no overlaps"))
    ax.legend(loc="upper left", fontsize=FS, ncol=2)
    finish(ax, "PC1 (projected)", "PC2 (projected)")

    # b
    ax = axes[1]
    lo, hi = np.percentile(x[coh], [5, 95])
    ax.axvspan(lo, hi, color=COHORT_COLOUR, alpha=0.16, lw=0, zorder=0)
    ax.axvline(float(np.median(x[coh])), color=COHORT_COLOUR, lw=1.0, zorder=1)
    _n_group = {g: sum(1 for i, s in enumerate(short) if s == g and ph[i])
                for g in groups}
    _gmin = {}
    for k, g in enumerate(groups):
        idx = [i for i, s in enumerate(short) if s == g and ph[i]]
        gx, gm = x[idx], np.nanmean(miss[idx])
        _gmin[k] = float(gx.min())
        ax.plot([gx.min(), gx.max()], [k, k], color="black", lw=1.0, zorder=2)
        ax.scatter([gx.mean()], [k], s=22, zorder=3, linewidths=0.6,
                   edgecolors="black",
                   facecolors=OKABE_ITO["orange"] if gm <= 0.5 else "none")
    ax.set_yticks(range(len(groups)))
    ax.set_yticklabels([f"{g} ({_n_group[g]})" for g in groups], fontsize=FS)
    for t, g in zip(ax.get_yticklabels(), groups):
        if g.startswith("Kazakh"):
            t.set_color(COHORT_COLOUR); t.set_fontweight("bold")
    ax.set_ylim(-0.8, len(groups) - 0.2)
    ax.set_xlim(*axes[0].get_xlim())
    _x0 = ax.get_xlim()[0]
    for k, xm in _gmin.items():
        ax.plot([_x0, xm], [k, k], ls=(0, (1, 2.5)), lw=0.5, color="0.72",
                zorder=0.5)
    ax.spines["left"].set_visible(True)
    finish(ax, "PC1 (projected)", None)
    ax.legend(handles=[
        Patch(facecolor=COHORT_COLOUR, alpha=0.16,
              label="cohort, 5th-95th percentile"),
        Line2D([0], [0], color=COHORT_COLOUR, lw=1.0, label="cohort median"),
        Line2D([0], [0], color="black", lw=1.0, marker="o", ms=4,
               mfc=OKABE_ITO["orange"], mec="black", mew=0.6,
               label="AADR group: mean and observed range"),
    ], loc="lower right", fontsize=FS, handlelength=1.6, borderpad=0.4)

    for a_, L in zip(axes, "ab"):
        a_.annotate(L, xy=(0, 1), xycoords="axes fraction",
                    xytext=(-30, 10), textcoords="offset points",
                    fontsize=FS_PANEL, fontweight="bold", va="bottom", ha="left")
    fig.subplots_adjust(left=0.20, right=0.98, top=0.93, bottom=0.075)
    _sd = pd.DataFrame({"Source": src, "Nation": short, "Population": pop,
                        "PC1": x, "PC2": y, "F_MISS": miss})
    source("SD_SFig4_AADR_projection", _sd)
    save(fig, "Supplementary_Fig_4_AADR_projection"); plt.show()

## Supplementary Data 2 - one Excel workbook

Only what a reader or reviewer would actually open. Tables that differ by one
column are merged rather than split across sheets: the two PC-association
tables become one with a `label` column, the two kinship tests likewise, and
the two cohort-capping checks share a sheet. Everything is centred, the header
row is bold, shaded and frozen, and columns are sized to their contents.

Sheets `SD_*` hold the source data behind every plotted panel (Communications
Biology asks for source data of all graphs); they carry no sample identifiers.

In [ ]:
import pandas as pd
from openpyxl.styles import Alignment, Font, PatternFill, Border, Side
from openpyxl.utils import get_column_letter

def cat(paths, key, values):
    out = []
    for v, src in zip(values, paths):
        if Path(src).exists() and Path(src).stat().st_size:
            d = pd.read_csv(src, sep="\t"); d.insert(0, key, v); out.append(d)
    return pd.concat(out, ignore_index=True) if out else None

def melt(items):
    rows = []
    for check, src in items:
        if not (Path(src).exists() and Path(src).stat().st_size):
            continue
        d = pd.read_csv(src, sep="\t")
        keycol = d.columns[0]
        for _, r in d.iterrows():
            for m in d.columns[1:]:
                rows.append({"check": check, "item": r[keycol],
                             "metric": m, "value": r[m]})
    return pd.DataFrame(rows) if rows else None

def one(src):
    return (pd.read_csv(src, sep="\t")
            if Path(src).exists() and Path(src).stat().st_size else None)

_FST_DIRS = sorted({d.name for g in (GA, GB) for d in (g/"fst").glob("*")
                    if d.is_dir()})
_FST_DIRS = [d for d in _FST_DIRS if d.endswith("_minn5")]

def fst_sheet(fname):
    paths, tags = [], []
    for g, gname in ((GA, "groupA"), (GB, "groupB")):
        for lv in _FST_DIRS:
            f = g/"fst"/lv/fname
            if f.exists():
                paths.append(f); tags.append(f"{gname} / {lv}")
    return cat(paths, "panel_level", tags)

print("  Fst runs going into the workbook: " + ", ".join(_FST_DIRS))

SHEETS = [
    ("1_sample_flow",     one(GB/"sample_flow.tsv")),
    ("2_Fst_vs_cohort",   fst_sheet("fst_vs_cohort.tsv")),
    ("3_Fst_all_pairs",   fst_sheet("fst_pairs.tsv")),
    ("4_FROH_by_pop",     one(GB/"roh/froh_by_population.tsv")),
    ("5_panel_QC",        one(RES/"meta/panel_qc.tsv")),
    ("6_PC_assoc",        cat([KZ/"pc_assoc_Tribe.tsv", KZ/"pc_assoc_Zhuz.tsv"],
                              "label", ["Tribe", "Zhuz"])),
    ("7_kinship_by_group",cat([KZ/"kinship_by_Tribe.tsv", KZ/"kinship_by_Zhuz.tsv"],
                              "label", ["Tribe", "Zhuz"])),
    ("8_robustness",      melt([
        ("relatedness cutoff", GB/"king_cutoff_sensitivity.tsv"),
    ] + [(f"relatedness cutoff, {'cohort-only panel' if 'kz_only' in f.parts else f.parent.relative_to(RES)}", f)
         for f in sorted(RES.rglob("king_cutoff_sensitivity.tsv")) if f != GB/"king_cutoff_sensitivity.tsv"
    ] + [
        ("ROH parameters",     RES/"kz_only/roh/sensitivity/froh_paramset_stats.tsv"),
        ("PCA capping, axes",  GB/"pca/pca_eurasia_capped.concordance.tsv"),
        ("PCA capping, space", GB/"pca/pca_eurasia_capped.subspace.tsv"),
        ("platform",           RES/"fst_platform_concordance/stats.tsv"),
    ] + [(f"ADMIXTURE capping, K={f.stem.rsplit('K', 1)[1]}", f) for f in
         sorted(GB.glob("admixture/projected_vs_joint.K*.tsv"))])),
    ("9_AADR_shrinkage",  one(GB/"pca_proj_aadr/matched_shrinkage.tsv")),
] + [(name, df) for name, df in SOURCE.items()]
print("  source-data sheets: " + (", ".join(SOURCE) if SOURCE else "none (figure cells not run)"))

xlsx, index = OUT/"Supplementary_Data_2_pca_admixture_fst_roh.xlsx", []
with pd.ExcelWriter(xlsx, engine="openpyxl") as xl:
    for name, df in SHEETS:
        if df is None or df.empty:
            print(f"  SKIP {name:<20} (stage not finished)"); continue
        if "rank" in df.columns:
            df = df.sort_values(["panel_level", "rank"])
        for c in ("POP", "POP1", "POP2", "population", "Population", "Nation"):
            if c in df.columns:
                df[c] = df[c].map(lambda v: DISPLAY_NAME.get(v, v))
        df.to_excel(xl, sheet_name=name[:31], index=False)
        index.append({"sheet": name, "rows": len(df), "columns": len(df.columns)})
        print(f"  {name:<20} {len(df):>5} rows x {len(df.columns)} cols")
    idx = pd.DataFrame(index)
    if DISPLAY_NAME:
        idx = pd.concat([idx, pd.DataFrame(
            [{"sheet": "display name", "rows": f"{k} -> {v}", "columns": ""}
             for k, v in DISPLAY_NAME.items()])], ignore_index=True)
    idx.to_excel(xl, sheet_name="INDEX", index=False)

    centre = Alignment(horizontal="center", vertical="center", wrap_text=False)
    head_fill = PatternFill(fill_type="solid", start_color="FFDDEBF7", end_color="FFDDEBF7")
    head_font = Font(name="Arial", size=10, bold=True, color="FF1F3864")
    body_font = Font(name="Arial", size=10)
    rule = Border(bottom=Side(style="thin", color="FFBFBFBF"))
    for ws in xl.book.worksheets:
        for row in ws.iter_rows():
            for c in row:
                c.alignment = centre
                c.font = body_font
        for c in ws[1]:
            c.font, c.fill, c.border = head_font, head_fill, rule
            c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        ws.row_dimensions[1].height = 30
        ws.freeze_panes = "A2"
        for j, col in enumerate(ws.columns, start=1):
            width = max((len(str(c.value)) for c in col if c.value is not None),
                        default=8)
            ws.column_dimensions[get_column_letter(j)].width = min(max(width+3, 10), 34)
print(f"\n-> {xlsx}  ({len(index)} sheets, all cells centred)")

## Numbers for the text

Every figure legend and every sentence of the Results needs a number, and the
one place those go wrong is being typed in by hand once and never updated. This
cell assembles them from the same tables the figures were drawn from, so the
legend and the panel cannot disagree. Copy from the output; do not retype from
a figure.

In [ ]:
ve    = load(GB/"pca/pca_eurasia.varexp.tsv",            "numbers", required=False)
conc  = load(GB/"pca/pca_eurasia_capped.concordance.tsv","numbers", required=False)
sub   = load(GB/"pca/pca_eurasia_capped.subspace.tsv",   "numbers", required=False)
cv    = load(GB/"admixture/cv_summary.tsv",              "numbers", required=False)
modes = load(GB/"admixture/train.modes_by_K.tsv",        "numbers", required=False)
rank  = load(GB/f"fst/{LEVEL}/fst_vs_cohort.tsv",        "numbers", required=False)
kzf   = load(RES/"kz_only/roh/froh_individual.tsv",      "numbers", required=False)
tfst  = load(KZ/"fst/Tribe/fst_pairs.tsv",               "numbers", required=False)
BEST_K = (int(cv["K"][int(np.nanargmin(nums(cv["CV_mean"])))]) if cv else None)

def get(tbl, key_col, key, val_col):
    if tbl is None or key_col not in tbl or val_col not in tbl: return None
    try:    return tbl[val_col][tbl[key_col].index(str(key))]
    except ValueError: return None

print("PANEL AND SAMPLE SIZES")
flow = load(GB/"sample_flow.tsv", "numbers", required=False)
if flow:
    for st, n in zip(flow["stage"], flow["samples"]):
        if st in ("merged_raw","unrelated_diploid","kazakh_unrelated","merged_pca",
                  "merged_admix","merged_admix_train","merged_fst"):
            print(f"  {st:<20} {n}")

print("\nPCA  (Fig. 2a,b)")
if ve is not None:
    for k in ("PC1","PC2"):
        i = ve["PC"].index(k)
        print(f"  {k} explains {float(ve['pct_variance'][i]):.2f}% of total variance")
if conc is not None:
    r12 = [float(v) for k, v in zip(conc["PC"], conc["abs_r"]) if k in ("PC1","PC2")]
    print(f"  cohort capped vs uncapped: |r| = {min(r12):.3f}-{max(r12):.3f} for PC1-PC2")
if sub is not None:
    i = len(sub["n_axes"]) - 1
    print(f"  subspace overlap over the first {sub['n_axes'][i]} axes: "
          f"{float(sub['subspace_overlap'][i]):.3f}, "
          f"largest principal angle {float(sub['max_principal_angle_deg'][i]):.1f} deg")

print("\nADMIXTURE  (Fig. 2c,d)")
if cv is not None and BEST_K:
    K, m, sd = nums(cv["K"]), nums(cv["CV_mean"]), nums(cv["CV_sd"])
    o = np.argsort(K); K, m, sd = K[o], m[o], sd[o]
    j = int(np.where(K == BEST_K)[0][0])
    print(f"  K = {BEST_K}: CV {m[j]:.5f} +/- {sd[j]:.5f} over {cv['n_runs'][j]} seeds")
    for nb_ in (j-1, j+1):
        if 0 <= nb_ < len(K):
            d = m[nb_] - m[j]; s_ = max(sd[j], sd[nb_], 1e-12)
            print(f"    vs K = {int(K[nb_])}: CV higher by {d:.5f} = {d/s_:.0f} SD")
if modes is not None and BEST_K:
    f = get(modes, "K", BEST_K, "major_mode_fraction")
    n = get(modes, "K", BEST_K, "n_runs")
    if f: print(f"  major mode at K = {BEST_K}: {float(f):.0%} of {n} replicates")
    multi = [(k, fr) for k, fr in zip(modes["K"], modes["major_mode_fraction"])
             if float(fr) < 1.0]
    if multi:
        print("    multimodal K: " + ", ".join(f"K={k} ({float(fr):.0%})"
                                               for k, fr in multi))
pvj = load(GB/f"admixture/projected_vs_joint.K{BEST_K}.tsv", "numbers",
           required=False) if BEST_K else None
if pvj is not None:
    r = np.abs(nums(pvj["pearson_r"]))
    print(f"  capped-and-projected vs uncapped joint fit: |r| = "
          f"{r.min():.3f}-{r.max():.3f} across {len(r)} components "
          f"(mean {r.mean():.4f})")

print("\nFst  (Supplementary Fig. 5, Supplementary Data 2)")
if rank is not None:
    fcol = next(c for c in rank if c.startswith("FST_vs_"))
    for i in range(min(3, len(rank["population"]))):
        print(f"  {i+1}. {rank['population'][i]:<12} "
              f"{float(rank[fcol][i]):.5f} "
              f"[{float(rank['CI95_LO'][i]):.5f}, {float(rank['CI95_HI'][i]):.5f}]"
              f"  separated from the previous: {rank['SEPARATED_FROM_PREVIOUS'][i]}")
pl = load(RES/"fst_platform_concordance/stats.tsv", "numbers", required=False)
if pl:
    for k, v in zip(pl["key"], pl["value"]):
        if k in ("pearson_r","slope_b_on_a","mean_diff_focus","mean_diff_other"):
            print(f"  platform check, {k}: {v}")

print("\nROH  (Fig. 3, Supplementary Table 4)")
if kzf is not None:
    tot, lng = nums(kzf["FROH"]), nums(kzf["FROH_LONG"])
    print(f"  cohort FROH  (runs >= 500 kb): {tot.mean():.4f} +/- {tot.std(ddof=1):.4f}")
    print(f"  cohort FROH  (runs >= 1.5 Mb): {lng.mean():.4f} +/- {lng.std(ddof=1):.4f}")
    print(f"  above 1/32 on the long-run measure: {int((lng > 1/32).sum())} "
          f"of {len(lng)} ({(lng > 1/32).mean():.1%})")

print("\nWITHIN THE COHORT  (Supplementary Fig. 3, Supplementary Table 3)")
if tfst is not None:
    for a, b, v, se, cz in zip(tfst["POP1"], tfst["POP2"], nums(tfst["HUDSON_FST"]),
                               nums(tfst["SE"]), tfst["CI_COVERS_ZERO"]):
        print(f"  {a} vs {b}: Fst {v:+.2e}, detectable >= {1.96*se:.2e}, "
              f"interval covers zero: {cz}")
for lv in ("Tribe", "Zhuz"):
    t_ = load(KZ/f"pc_assoc_{lv}.tsv", "numbers", required=False)
    if t_:
        pb = nums(t_["p_bonferroni"]); i = int(np.nanargmin(pb))
        print(f"  {lv}: smallest PC association {t_['PC'][i]}, "
              f"p_perm {t_['p_perm'][i]}, Bonferroni {t_['p_bonferroni'][i]}")
    k_ = load(KZ/f"kinship_by_{lv}.tsv", "numbers", required=False)
    if k_:
        d = dict(zip(k_["key"], k_["value"]))
        print(f"  {lv}: kinship within {d.get('mean_kinship_within')} vs between "
              f"{d.get('mean_kinship_between')}, p = {d.get('p_two_sided_permutation')}")

print("\nAADR PROJECTION  (Supplementary Fig. 4)")
ms = load(GB/"pca_proj_aadr/matched_shrinkage.tsv", "numbers", required=False)
if ms:
    def col(name, i, dash="-"):
        return ms[name][i] if name in ms and i < len(ms[name]) else dash
    for i, g in enumerate(ms["pseudohaploid_group"]):
        extra = (f" [{col('ratio_CI_LO', i)}, {col('ratio_CI_HI', i)}], "
                 f"covers 1: {col('RATIO_CI_COVERS_1', i)}") if "ratio_CI_LO" in ms \
                else "  (no interval: rerun 08 with the current popgen_tables.py)"
        print(f"  {g} vs {col('diploid_group', i)}: norm ratio "
              f"{col('norm_ratio', i)}{extra}, angle {col('angle_deg', i)} deg")

## Supplementary Tables 2-6 - the SI docx

Only the tables a reader needs next to the text; everything else stays in the
workbook. The docx is built from the same `SHEETS` frames that were just
written to Excel, so a value cannot differ between the two. Long-form sheets
are reshaped rather than pasted: the two key-value kinship tests become one row
per stratum, the PC associations one row per axis with a Tribe | Zhuz column
group, and `8_robustness` is split into its checks, each a short wide block.
Tables of different shape are never joined side by side; they sit under one
caption as (a), (b), (c). The last lines reconcile every number in the saved
docx back to the workbook and print whatever is not traceable.

- `NUM` continues from notebook 01 (Supplementary Table 1) in order of first citation:
  2 relatedness, 3 structure, 4 F_ROH (Results), 5 reference panels and 6 robustness (Methods).
  In v24 the F_ROH table was 5 and the panel table 4.
- Tables carry horizontal rules only, no shading; Arial; missing values are "-".
- Parts (b)-(d) of the relatedness table are read from the KING output
  (`*.kin0`, `*.king.cutoff.out.id`) if it sits under `results/`; a cohort-only
  `king_cutoff_sensitivity.tsv` there adds the 233-panel columns to (a) and is
  melted into `8_robustness` as well.
- The two F_ST tables are workbook-only (72 and 1,070 rows; Supplementary Fig. 5 shows them).

In [ ]:
import re
from docx import Document
from docx.shared import Pt, Mm
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.enum.section import WD_ORIENT
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

SH = {name: df for name, df in SHEETS if df is not None and not df.empty}

NUM = {"relatedness": 2, "structure": 3, "froh": 4, "panels": 5, "robustness": 6}
ANONYMIZE = True 

DEGREE = {0.354: "duplicate or monozygotic", 0.177: "first-degree and closer",
          0.0884: "second-degree and closer", 0.0442: "third-degree and closer"}

def fmt(v, d=None):
    if v is None or v == "-" or (isinstance(v, float) and np.isnan(v)):
        return "-"
    if isinstance(v, str):
        try: v = float(v)
        except ValueError: return v
    if isinstance(v, (bool, np.bool_)):
        return "yes" if v else "no"
    v = float(v)
    if d is None:
        return f"{int(v):,}" if v.is_integer() else f"{v:g}"
    return f"{v:,.{d}f}"

def pfmt(v):
    v = float(v)
    if v == 0:    return "<1e-4"
    if v >= 1:    return "1.00"
    if v >= 0.01: return f"{v:.2f}"
    return f"{v:.3f}" if v >= 0.001 else f"{v:.1e}"

def order(seq):
    return list(dict.fromkeys(seq))

def kinship_wide(df):
    out = []
    for lab in order(df["label"]):
        d = dict(zip(df.loc[df.label == lab, "key"], df.loc[df.label == lab, "value"]))
        out.append({"Stratum": lab, "n": fmt(d["n_individuals"]),
                    "Levels": str(d["levels"]).replace(",", ", "),
                    "Pairs within / between": f'{fmt(d["n_pairs_within"])} / {fmt(d["n_pairs_between"])}',
                    "Mean φ within": fmt(d["mean_kinship_within"], 5),
                    "Mean φ between": fmt(d["mean_kinship_between"], 5),
                    "Δ within - between": f'{float(d["difference_within_minus_between"]):+.5f}',
                    "Median φ, all pairs (IQR)": f'{d["median_kinship_all"]} ({d["iqr_kinship_all"]})',
                    f'P (two-sided, {fmt(d["n_permutations"])} permutations)': pfmt(d["p_two_sided_permutation"])})
    return pd.DataFrame(out)

def pc_assoc_wide(df):
    labs, pcs = order(df["label"]), order(df["PC"])
    head = {lab: f'{lab} (n = {fmt(df.loc[df.label == lab, "n"].iloc[0])}, '
                 f'{fmt(df.loc[df.label == lab, "n_levels"].iloc[0])} levels)' for lab in labs}
    rows = []
    for pc in pcs:
        r = {("Axis", ""): pc}
        for lab in labs:
            x = df[(df.label == lab) & (df.PC == pc)].iloc[0]
            r[(head[lab], "F")] = fmt(x["F"], 3)
            r[(head[lab], "P (permutation)")] = pfmt(x["p_perm"])
            r[(head[lab], "P (Bonferroni)")] = pfmt(x["p_bonferroni"])
        rows.append(r)
    out = pd.DataFrame(rows); out.columns = pd.MultiIndex.from_tuples(out.columns)
    return out

def wide(df, check):
    g = df[df["check"] == check]
    if g.empty:
        return None
    if g.duplicated(["item", "metric"]).any():
        print(f"  WARNING {check}: duplicated item/metric pairs, first kept")
    return (g.pivot_table(index="item", columns="metric", values="value", aggfunc="first", sort=False)
             .reindex(order(g["item"])))

def relatedness_table(df):
    checks = [c for c in order(df["check"]) if str(c).startswith("relatedness cutoff")]
    if not checks:
        return None
    parts = {c: wide(df, c) for c in checks}
    cut = sorted({float(i) for w in parts.values() for i in w.index})
    rows = []
    for c in cut:
        r = {"Kinship threshold (φ >)": f"{c:g}",
             "Pairs removed": next((v for k, v in DEGREE.items() if abs(k - c) < 1e-9), "-")}
        for chk, w in parts.items():
            lab = "merged panel" if chk == "relatedness cutoff" else chk.split(",", 1)[1].strip()
            hit = w.loc[[i for i in w.index if abs(float(i) - c) < 1e-9]]
            for m, name in (("n_retained", "retained"), ("n_removed", "removed"), ("n_kazakh", "Kazakh retained")):
                if m in w.columns:
                    r[f"{name.capitalize()}, {lab}"] = fmt(hit[m].iloc[0]) if len(hit) else "-"
        rows.append(r)
    return pd.DataFrame(rows)

def robustness_blocks(df):
    b = []
    w = wide(df, "ROH parameters")
    if w is not None:
        name = {"wgs": "Default (reported)", "lenient": "Lenient", "long_only": "Long runs only",
                "nomaf": "No MAF filter", "strict": "Strict"}
        b.append(("(a) Within-cohort F_ROH under alternative run-calling parameters, against the default setting",
                  pd.DataFrame({"Setting": [name.get(i, i) for i in w.index], "n": [fmt(v) for v in w["n"]],
                                "F_ROH mean": [fmt(v, 4) for v in w["FROH_mean"]],
                                "F_ROH median": [fmt(v, 4) for v in w["FROH_median"]],
                                "Pearson r vs default": [fmt(v, 3) for v in w["pearson_vs_wgs"]],
                                "Spearman ρ vs default": [fmt(v, 3) for v in w["spearman_vs_wgs"]]})))
    w = wide(df, "PCA capping, axes")
    if w is not None:
        b.append(("(b) Principal components with the cohort capped at 25 individuals against the uncapped axes, per axis",
                  pd.DataFrame({"Axis": list(w.index), "Shared individuals": [fmt(v) for v in w["n_shared"]],
                                "|r|": [fmt(v, 3) for v in w["abs_r"]]})))
    w = wide(df, "PCA capping, space")
    if w is not None:
        b.append(("(c) The same comparison as a subspace",
                  pd.DataFrame({"Subspace": [f"first {i} axes" for i in w.index],
                                "Overlap": [fmt(v, 3) for v in w["subspace_overlap"]],
                                "Largest principal angle (°)": [fmt(v, 1) for v in w["max_principal_angle_deg"]],
                                "Principal angles (°)": [str(v).replace(" ", ", ") for v in w["principal_angles_deg"]]})))
    for chk in [c for c in order(df["check"]) if str(c).startswith("ADMIXTURE capping")]:
        w = wide(df, chk)
        b.append((f"(d) ADMIXTURE at {chk.split(',')[-1].strip()}, capped-and-projected fit against the uncapped joint fit, component by component",
                  pd.DataFrame({"Component (capped fit)": list(w.index),
                                "Matched component (joint fit)": [f"Q{fmt(v)}" for v in w["matched_component_B"]],
                                "Individuals": [fmt(v) for v in w["n_shared"]],
                                "Pearson r": [fmt(v, 4) for v in w["pearson_r"]],
                                "Mean |Δ|": [fmt(v, 4) for v in w["mean_abs_diff"]]})))
    w = wide(df, "platform")
    if w is not None:
        d = w["value"].to_dict()
        b.append(("(e) Platform concordance of superpopulation F_ST between the two reference panels",
                  pd.DataFrame([{"Shared superpopulation pairs": fmt(d["n_shared_pairs"]),
                                 "Pearson r": fmt(d["pearson_r"], 3),
                                 "Slope (HGDP on 1000 Genomes)": fmt(d["slope_b_on_a"], 3),
                                 "Mean |Δ|": fmt(d["mean_abs_diff"], 4),
                                 "Mean Δ, pairs including the cohort": fmt(d["mean_diff_focus"], 4),
                                 "Mean Δ, other pairs": fmt(d["mean_diff_other"], 4)}])))
    return b

def panel_qc(df):
    name = {"kz": "KZ-219 (this study)", "kg": "1000 Genomes Phase 3", "sgdp": "SGDP", "hgdp": "HGDP", "aadr": "AADR v62, 1240K"}
    return pd.DataFrame({"Panel": [name.get(p, p) for p in df["panel"]],
                         "Platform": df["platform"].str.upper().str.replace("ARRAY", "array").str.replace("CAPTURE", "capture"),
                         "Samples": [fmt(v) for v in df["samples"]], "Variants": [fmt(v) for v in df["variants"]],
                         "Duplicate IDs": [fmt(v) for v in df["dup_ids"]],
                         "Strand-ambiguous sites removed": [fmt(v) for v in df["ambiguous"]],
                         "Heterozygosity outliers dropped / flagged": [f"{fmt(a)} / {fmt(b)}" for a, b in zip(df["het_dropped"], df["het_flagged"])],
                         "Missingness outliers dropped": [fmt(v) for v in df["mind_dropped"]]})

def sample_flow(df):
    name = {"merged_raw": "Merged callset, all panels", "pseudohaploid": "Pseudohaploid samples",
            "king_markers": "KING marker set", "unrelated_diploid": "Unrelated diploid individuals",
            "kazakh_unrelated": "Unrelated Kazakh individuals", "pca_capped_ref": "PCA: reference populations capped at 25",
            "pca_capped_both": "PCA: reference and cohort capped", "merged_pca": "PCA input",
            "merged_admix": "ADMIXTURE projection panel", "merged_admix_train": "ADMIXTURE training panel",
            "merged_fst": "F_ST panel"}
    return pd.DataFrame({"Stage": [name.get(s, s) for s in df["stage"]], "Key in Supplementary Data 2": [f"({s})" for s in df["stage"]],
                         "Samples": [fmt(v) for v in df["samples"]], "Markers": [fmt(v) for v in df["variants"]]})

def froh(df):
    return pd.DataFrame({"Population": df["Population"], "n": [fmt(v) for v in df["n"]],
                         "F_ROH mean": [fmt(v, 4) for v in df["FROH_mean"]], "F_ROH median": [fmt(v, 4) for v in df["FROH_median"]],
                         "SD": [fmt(v, 4) for v in df["FROH_sd"]], "Min": [fmt(v, 4) for v in df["FROH_min"]], "Max": [fmt(v, 4) for v in df["FROH_max"]],
                         "F_ROH, long runs, mean": [fmt(v, 4) for v in df["FROH_LONG_mean"]],
                         "F_ROH, long runs, median": [fmt(v, 4) for v in df["FROH_LONG_median"]],
                         "Fraction above 1/32": [fmt(v, 3) for v in df["frac_FROH_LONG_gt_0.03125"]]})

KIN0    = sorted(RES.rglob("*.kin0"), key=lambda f: ("kz" not in str(f.relative_to(RES)).lower(), str(f)))
REMOVED = sorted(RES.rglob("*.king.cutoff.out.id"), key=lambda f: ("kz" not in str(f.relative_to(RES)).lower(), str(f)))
KIN0, REMOVED = (KIN0[0] if KIN0 else None), (REMOVED[0] if REMOVED else None)
pairs_by_degree = pairs_listed = removed_tbl = None
king_vals = set()
if KIN0 is not None:
    k = pd.read_csv(KIN0, sep=r"\s+")
    k.columns = [c.lstrip("#") for c in k.columns]
    kc = next(c for c in k.columns if c.upper() == "KINSHIP")
    i1 = next(c for c in k.columns if c.upper() in ("IID1", "ID1")); i2 = next(c for c in k.columns if c.upper() in ("IID2", "ID2"))
    rem = (pd.read_csv(REMOVED, sep=r"\s+").iloc[:, -1].astype(str).tolist() if REMOVED is not None else [])
    ids = sorted(set(k[i1].astype(str)) | set(k[i2].astype(str)) | set(rem))
    amap = {x: f"P{j+1:03d}" for j, x in enumerate(ids)} if ANONYMIZE else {x: x for x in ids}
    if ANONYMIZE:
        pd.DataFrame({"participant": list(amap.values()), "sample": list(amap.keys())}).to_csv(
            OUT/"participant_id_map.tsv", sep="\t", index=False)
    kin = k[kc].astype(float)
    def degree(v):
        return next((d for t, d in DEGREE.items() if v > t), "unrelated")
    bins = [("> 0.354", "duplicate or monozygotic", (kin > 0.354).sum()),
            ("0.177-0.354", "first-degree", ((kin > 0.177) & (kin <= 0.354)).sum()),
            ("0.0884-0.177", "second-degree", ((kin > 0.0884) & (kin <= 0.177)).sum()),
            ("0.0442-0.0884", "third-degree", ((kin > 0.0442) & (kin <= 0.0884)).sum()),
            ("≤ 0.0442", "unrelated", (kin <= 0.0442).sum())]
    nsnp = next((c for c in k.columns if c.upper() in ("NSNP", "N_SNP")), None)
    pairs_by_degree = pd.DataFrame(
        [{"Quantity": "Individuals", "Value": fmt(len(set(k[i1].astype(str)) | set(k[i2].astype(str))))},
         {"Quantity": "Markers (median per pair)", "Value": fmt(k[nsnp].median()) if nsnp else "-"},
         {"Quantity": "Pairs listed", "Value": fmt(len(k))}] +
        [{"Quantity": f"Pairs at kinship {rng_}, {lab}", "Value": fmt(int(n))} for rng_, lab, n in bins])
    close = k[kin > 0.0442].sort_values(kc, ascending=False)
    pairs_listed = pd.DataFrame({"Participant 1": close[i1].astype(str).map(amap), "Participant 2": close[i2].astype(str).map(amap),
                                 "Kinship φ": [fmt(v, 4) for v in close[kc]],
                                 "Inferred degree": [degree(float(v)) for v in close[kc]]})
    if rem:
        removed_tbl = pd.DataFrame({"Participant": [amap[x] for x in rem],
                                    "Closest pair, kinship φ": [fmt(kin[(k[i1].astype(str) == x) | (k[i2].astype(str) == x)].max(), 4) for x in rem]})
    king_vals = (set(kin.round(4)) | {float(n) for *_, n in bins} | {len(k), len(set(k[i1].astype(str)) | set(k[i2].astype(str)))}
                 | ({float(k[nsnp].median())} if nsnp else set()))
    print(f"  KING pairs from {KIN0.relative_to(RES)}" + (f", removed ids from {REMOVED.relative_to(RES)}" if REMOVED else ", no *.king.cutoff.out.id found"))
else:
    print("  no *.kin0 under results: parts (b)-(d) of the relatedness table are skipped")

rob = SH.get("8_robustness")
SUPPL_TABLES = [
    ("relatedness", "Relatedness.",
     [("(a) Genomes retained at each kinship threshold", relatedness_table(rob) if rob is not None else None),
      ("(b) Marker count and pairs by inferred degree", pairs_by_degree),
      ("(c) Every pair above the third-degree threshold (φ > 0.0442)", pairs_listed),
      ("(d) Genomes removed by the reported filter", removed_tbl)],
     ["Kinship φ from KING; degrees follow the KING intervals. Participant identifiers are anonymized; the map is kept with the analysis files and not deposited."]),

    ("structure", "Tests for genetic structure by self-reported tribe and juz.",
     [("(a) Mean kinship within and between strata, permutation test", kinship_wide(SH["7_kinship_by_group"]) if "7_kinship_by_group" in SH else None),
      ("(b) Permutation association of the first ten within-cohort principal components with stratum, Bonferroni-corrected across the ten axes",
       pc_assoc_wide(SH["6_PC_assoc"]) if "6_PC_assoc" in SH else None)],
     ["Source: Supplementary Data 2, sheets 7_kinship_by_group and 6_PC_assoc. Strata with fewer than ten members were excluded from both tests; individuals without a recorded affiliation are not included in n."]),

    ("froh", "Runs of homozygosity: F_ROH by population and run-length class.",
     [(None, froh(SH["4_FROH_by_pop"]) if "4_FROH_by_pop" in SH else None)],
     ["Source: Supplementary Data 2, sheet 4_FROH_by_pop. Long runs are those of at least 1.5 Mb; the last column is the fraction of individuals whose long-run F_ROH exceeds 1/32."]),

    ("panels", "Assembly of the reference panels.",
     [("(a) Quality control of each source panel before merging", panel_qc(SH["5_panel_QC"]) if "5_panel_QC" in SH else None),
      ("(b) Samples and markers entering each analysis stage", sample_flow(SH["1_sample_flow"]) if "1_sample_flow" in SH else None)],
     ["Source: Supplementary Data 2, sheets 5_panel_QC and 1_sample_flow. Marker counts are after LD pruning where the stage is an analysis input."]),

    ("robustness", "Robustness of the population-structure analyses to parameter choices.",
     robustness_blocks(rob) if rob is not None else [],
     ["Source: Supplementary Data 2, sheet 8_robustness, which keeps every value in long form. In (e), a cohort-specific offset larger than that of the other pairs would be the signature of a platform effect on the cohort; the observed offset is smaller."]),
]

FONT, USABLE_MM = "Arial", 297 - 2 * 18

def _write(cell, text, size, bold=False, italic=False, align=WD_ALIGN_PARAGRAPH.CENTER):
    cell.text = ""
    p = cell.paragraphs[0]; p.alignment = align
    p.paragraph_format.space_before = p.paragraph_format.space_after = Pt(0)
    r = p.add_run(text); r.font.size = Pt(size); r.font.name = FONT; r.bold = bold; r.italic = italic

def _rules(t, nh):
    tblPr = t._tbl.tblPr
    b = OxmlElement("w:tblBorders")
    for side, val in (("top", "single"), ("bottom", "single"), ("left", "nil"), ("right", "nil"), ("insideH", "nil"), ("insideV", "nil")):
        e = OxmlElement(f"w:{side}"); e.set(qn("w:val"), val)
        if val == "single": e.set(qn("w:sz"), "8"); e.set(qn("w:color"), "000000")
        b.append(e)
    tblPr.append(b)
    for c in t.rows[nh - 1].cells:
        tb = OxmlElement("w:tcBorders"); bt = OxmlElement("w:bottom")
        bt.set(qn("w:val"), "single"); bt.set(qn("w:sz"), "4"); bt.set(qn("w:color"), "000000")
        tb.append(bt); c._tc.get_or_add_tcPr().append(tb)
    for r in t.rows[:nh]:
        h = OxmlElement("w:tblHeader"); h.set(qn("w:val"), "true"); r._tr.get_or_add_trPr().append(h)

def _widths(df):
    cols = [c[1] or c[0] for c in df.columns] if isinstance(df.columns, pd.MultiIndex) else [str(c) for c in df.columns]
    w = []
    for j, c in enumerate(cols):
        vals = [c] + [str(v) for v in df.iloc[:, j]]
        w.append(max(6, min(22, max(len(v) for v in vals)), min(24, max(len(t) for v in vals for t in v.split()) + 2)))
    tot = sum(w); table_mm = min(USABLE_MM, tot * 1.9 + len(w) * 3.0)
    return [Mm(table_mm * x / tot) for x in w]

def add_block(doc, df, sub=None):
    if sub:
        p = doc.add_paragraph(); r = p.add_run(sub); r.italic = True; r.font.size = Pt(8); r.font.name = FONT
        p.paragraph_format.space_before = Pt(6); p.paragraph_format.space_after = Pt(2)
    two = isinstance(df.columns, pd.MultiIndex)
    nh = 2 if two else 1
    size = 7.5 if sum(len(str(v)) for v in df.iloc[0]) + 2 * df.shape[1] > 120 else 8
    t = doc.add_table(rows=nh, cols=df.shape[1]); t.alignment = WD_TABLE_ALIGNMENT.CENTER
    widths = _widths(df)
    cols = list(df.columns)
    if two:
        j = 0
        while j < len(cols):
            k = j
            while k + 1 < len(cols) and cols[k + 1][0] == cols[j][0]: k += 1
            if k > j:
                _write(t.cell(0, j).merge(t.cell(0, k)), cols[j][0], size, True)
                for jj in range(j, k + 1): _write(t.cell(1, jj), cols[jj][1], size, True)
            else:
                _write(t.cell(0, j).merge(t.cell(1, j)), cols[j][0], size, True)
            j = k + 1
    else:
        for j, c in enumerate(cols): _write(t.cell(0, j), str(c), size, True)
    for _, row in df.iterrows():
        cells = t.add_row().cells
        for j, v in enumerate(row):
            s = v if isinstance(v, str) else fmt(v)
            num = s[:1].isdigit() or s[:1] in "−+-<"
            _write(cells[j], s, size, align=WD_ALIGN_PARAGRAPH.CENTER if (j > 0 and num) else WD_ALIGN_PARAGRAPH.LEFT)
    for row in t.rows:
        for j, c in enumerate(row.cells): c.width = widths[min(j, len(widths) - 1)]
    _rules(t, nh)

doc = Document()
s = doc.sections[0]
s.orientation = WD_ORIENT.LANDSCAPE; s.page_width, s.page_height = Mm(297), Mm(210)
s.left_margin = s.right_margin = s.top_margin = s.bottom_margin = Mm(18)

written = 0
for key, cap, blocks, notes in SUPPL_TABLES:
    blocks = [(sub, df) for sub, df in blocks if df is not None and not df.empty]
    if not blocks:
        print(f"  Supplementary Table {NUM[key]} ({key}): nothing available, skipped"); continue
    if written: doc.add_page_break()
    p = doc.add_paragraph(); p.paragraph_format.space_after = Pt(4)
    r = p.add_run(f"Supplementary Table {NUM[key]}. {cap}"); r.bold = True; r.font.size = Pt(9.5); r.font.name = FONT
    for sub, df in blocks: add_block(doc, df, sub if len(blocks) > 1 else None)
    for n_ in notes:
        p = doc.add_paragraph(); p.paragraph_format.space_before = Pt(5)
        r = p.add_run(n_); r.font.size = Pt(7.5); r.font.name = FONT
    written += 1
    print(f"  Supplementary Table {NUM[key]} ({key}): " + "; ".join(f"{df.shape[0]}x{df.shape[1]}" for _, df in blocks))

DOCX = OUT / "Supplementary_Tables_02.docx"
doc.save(DOCX)
print(f"\nsaved {DOCX.name}: {written} tables")

src = set(king_vals)
for df in SH.values():
    for v in df.values.ravel():
        try: src.add(float(v))
        except (TypeError, ValueError):
            for tok in re.findall(r"\d+\.?\d*", str(v)): src.add(float(tok))
src |= set(DEGREE) | {0.354, 0.177, 25, 10, 32}      # thresholds and constants that appear in captions
SRC = np.array(sorted(src)); _rounded = {}
def _seen(tok):
    x = float(tok.replace(",", "")); d = len(tok.split(".")[1]) if "." in tok else 0
    R = _rounded.setdefault(d, np.round(SRC, d))
    return bool(np.any(np.isclose(R, x, atol=10.0 ** -(d + 6)))) or bool(np.any(np.isclose(SRC, x, atol=10.0 ** -(d + 6))))
checked, missing = 0, []
for t in Document(DOCX).tables:
    for row in t.rows:
        for c in row.cells:
            s = c.text
            if re.search(r"\d_[A-Za-z0-9]|^P\d{3}$|first \d axes|Q\d$|^PC\d+$|\d+ Genomes|\d+K\b|v\d+", s):    # ids and labels
                continue
            for tok in re.findall(r"(?<![A-Za-z\d.])-?\d[\d,]*\.?\d*(?![\d,]*e)", s):
                checked += 1
                if not _seen(tok): missing.append((tok, s))
print(f"reconciled {checked} numbers against the workbook: {len(missing)} not found" + ("" if not missing else ":"))
for tok, s in missing[:40]:
    print(f"  {tok:>12}  in  '{s}'")


## Outputs

In [ ]:
for pat, what in (("*.tiff", "figures, TIFF (CMYK)"), ("*.png", "figures, PNG (RGB)"),
                  ("*.xlsx", "workbook"), ("*.docx", "SI tables")):
    files = sorted(p.name for p in OUT.glob(pat))
    print(f"{what:<22} {len(files)}: " + ", ".join(files))